# ****Data_prep file****

In [1]:
%%writefile data_prep.py
"""
Shared data prep for the DDP / FSDP / JAX benchmark.
Tokenizes a subsample of tatsu-lab/alpaca with the Qwen2.5-1.5B-Instruct tokenizer,
caps sequence length, and saves as a fixed-size tensor file so all three training
scripts train on IDENTICAL data (no framework-specific randomness in what's seen).
"""

import argparse
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

ALPACA_PROMPT = """### Instruction:
{instruction}
{input_block}
### Response:
{output}"""


def format_example(ex):
    input_block = f"\n### Input:\n{ex['input']}" if ex["input"].strip() else ""
    return ALPACA_PROMPT.format(
        instruction=ex["instruction"], input_block=input_block, output=ex["output"]
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-examples", type=int, default=3000,
                         help="how many Alpaca examples to use (benchmark, not full training)")
    parser.add_argument("--max-length", type=int, default=512)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out", type=str, default="tokenized_alpaca.npz")
    args = parser.parse_args()

    print(f"Loading tokenizer: {MODEL_NAME}")
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    print("Loading tatsu-lab/alpaca ...")
    ds = load_dataset("tatsu-lab/alpaca", split="train")
    ds = ds.shuffle(seed=args.seed).select(range(args.n_examples))

    texts = [format_example(ex) for ex in ds]

    print(f"Tokenizing {len(texts)} examples, max_length={args.max_length} ...")
    enc = tok(
        texts,
        max_length=args.max_length,
        truncation=True,
        padding="max_length",
        return_tensors="np",
    )

    input_ids = enc["input_ids"].astype(np.int32)
    attention_mask = enc["attention_mask"].astype(np.int32)

    # Causal LM labels = input_ids, with padding masked to -100
    labels = input_ids.copy()
    labels[attention_mask == 0] = -100

    np.savez(
        args.out,
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )
    print(f"Saved {input_ids.shape[0]} examples of length {input_ids.shape[1]} to {args.out}")
    print(f"Shapes -> input_ids: {input_ids.shape}, attention_mask: {attention_mask.shape}, labels: {labels.shape}")


if __name__ == "__main__":
    main()

Writing data_prep.py


# PyTorch DDP

In [2]:
%%writefile pytorch_ddp.py
"""
PyTorch DDP full-fine-tuning baseline for Qwen2.5-1.5B-Instruct on Alpaca.

Launch with:
    torchrun --nproc_per_node=<N_GPUS> train_pytorch_ddp.py \
        --data tokenized_alpaca.npz --steps 100

Logs per-step throughput and memory to results_ddp.json in the SAME schema
used by train_pytorch_fsdp.py and train_jax.py, so analysis.ipynb can load
all three without framework-specific parsing.
"""

import os
import json
import time
import argparse

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# ---- shared logging schema across ddp/fsdp/jax scripts ----
# {step, timestamp, tokens_per_sec, peak_memory_mb, loss}


class TokenizedDataset(Dataset):
    def __init__(self, npz_path):
        data = np.load(npz_path)
        self.input_ids = torch.from_numpy(data["input_ids"]).long()
        self.attention_mask = torch.from_numpy(data["attention_mask"]).long()
        self.labels = torch.from_numpy(data["labels"]).long()

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


def setup_distributed():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank, dist.get_rank(), dist.get_world_size()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, default="tokenized_alpaca.npz")
    parser.add_argument("--batch-size", type=int, default=1, help="per-GPU micro batch size")
    parser.add_argument("--steps", type=int, default=100)
    parser.add_argument("--lr", type=float, default=1e-5)
    parser.add_argument("--warmup-steps", type=int, default=10, help="excluded from throughput stats")
    parser.add_argument("--out", type=str, default="results_ddp.json")
    args = parser.parse_args()

    local_rank, rank, world_size = setup_distributed()
    device = torch.device(f"cuda:{local_rank}")

    if rank == 0:
        print(f"World size: {world_size}, loading model {MODEL_NAME} in bf16 ...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
    ).to(device)

    model = DDP(model, device_ids=[local_rank])
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)

    dataset = TokenizedDataset(args.data)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler, drop_last=True)

    seq_len = dataset.input_ids.shape[1]
    tokens_per_step = args.batch_size * world_size * seq_len

    logs = []
    step = 0
    data_iter = iter(loader)

    torch.cuda.reset_peak_memory_stats(device)
    model.train()

    while step < args.steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            sampler.set_epoch(step)  # reshuffle
            data_iter = iter(loader)
            batch = next(data_iter)

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        torch.cuda.synchronize()
        t0 = time.perf_counter()

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        torch.cuda.synchronize()
        t1 = time.perf_counter()

        step += 1
        step_time = t1 - t0

        if step > args.warmup_steps and rank == 0:
            peak_mem_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
            logs.append({
                "step": step,
                "timestamp": time.time(),
                "step_time_sec": step_time,
                "tokens_per_sec": tokens_per_step / step_time,
                "peak_memory_mb": peak_mem_mb,
                "loss": loss.item(),
            })
            if step % 10 == 0:
                print(f"[step {step}] loss={loss.item():.4f} "
                      f"tok/s={tokens_per_step / step_time:.1f} "
                      f"peak_mem={peak_mem_mb:.0f}MB")

    if rank == 0:
        with open(args.out, "w") as f:
            json.dump({
                "framework": "pytorch_ddp",
                "world_size": world_size,
                "seq_len": seq_len,
                "per_gpu_batch_size": args.batch_size,
                "warmup_steps_excluded": args.warmup_steps,
                "logs": logs,
            }, f, indent=2)
        print(f"Saved results to {args.out}")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing pytorch_ddp.py


# PyTorch FSDP

In [3]:
%%writefile pytorch_fsdp.py
"""
PyTorch FSDP full-fine-tuning baseline for Qwen2.5-1.5B-Instruct on Alpaca.

Launch with:
    torchrun --nproc_per_node=<N_GPUS> train_pytorch_fsdp.py \
        --data tokenized_alpaca.npz --steps 100

Same logging schema as train_pytorch_ddp.py / train_jax.py so analysis.ipynb
can load all three results files without framework-specific parsing:
    {step, timestamp, step_time_sec, tokens_per_sec, peak_memory_mb, loss}

Key difference from DDP: instead of replicating the full model + optimizer
state on every GPU, FSDP shards params/gradients/optimizer state across
GPUs, gathering full weights per-layer only transiently during forward/
backward. This is what makes full fine-tuning of a 1.5B model feasible on
GPUs that couldn't hold the full optimizer state individually.
"""

import os
import json
import time
import argparse
import functools

import numpy as np
import torch
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from transformers import AutoModelForCausalLM
from transformers.models.qwen2.modeling_qwen2 import Qwen2DecoderLayer

from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


class TokenizedDataset(Dataset):
    def __init__(self, npz_path):
        data = np.load(npz_path)
        self.input_ids = torch.from_numpy(data["input_ids"]).long()
        self.attention_mask = torch.from_numpy(data["attention_mask"]).long()
        self.labels = torch.from_numpy(data["labels"]).long()

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


def setup_distributed():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank, dist.get_rank(), dist.get_world_size()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, default="tokenized_alpaca.npz")
    parser.add_argument("--batch-size", type=int, default=1, help="per-GPU micro batch size")
    parser.add_argument("--steps", type=int, default=100)
    parser.add_argument("--lr", type=float, default=1e-5)
    parser.add_argument("--warmup-steps", type=int, default=10, help="excluded from throughput stats")
    parser.add_argument("--out", type=str, default="results_fsdp.json")
    parser.add_argument(
        "--sharding-strategy",
        type=str,
        default="FULL_SHARD",
        choices=["FULL_SHARD", "SHARD_GRAD_OP", "HYBRID_SHARD"],
        help="FULL_SHARD = params+grads+optim all sharded (closest JAX-sharding analog). "
             "SHARD_GRAD_OP = only grads+optim sharded, params replicated (more like DDP+savings).",
    )
    args = parser.parse_args()

    local_rank, rank, world_size = setup_distributed()
    device = torch.device(f"cuda:{local_rank}")

    if rank == 0:
        print(f"World size: {world_size}, loading model {MODEL_NAME} in bf16 ...")
        print(f"Sharding strategy: {args.sharding_strategy}")

    # Load on CPU first / meta-ish to avoid every rank materializing a full
    # bf16 copy on GPU before sharding kicks in (matters more at larger scale;
    # kept simple here since 1.5B is small enough to load directly).
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )

    # Wrap each Qwen2 decoder layer as its own FSDP unit. This is the standard
    # "auto wrap" pattern -- it determines the granularity at which params are
    # sharded/gathered. Wrapping at the transformer-block level (rather than
    # the whole model as one FSDP unit) is what allows the per-layer
    # gather-compute-discard cycle that keeps peak memory low.
    auto_wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={Qwen2DecoderLayer},
    )

    mixed_precision_policy = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    )

    sharding_strategy = getattr(ShardingStrategy, args.sharding_strategy)

    model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        mixed_precision=mixed_precision_policy,
        sharding_strategy=sharding_strategy,
        device_id=local_rank,
    )

    # NOTE: optimizer is constructed AFTER FSDP-wrapping the model, so the
    # optimizer only ever sees (and allocates state for) this rank's shard
    # of the parameters -- this is where FSDP's optimizer-state memory
    # savings actually come from.
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)

    dataset = TokenizedDataset(args.data)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler, drop_last=True)

    seq_len = dataset.input_ids.shape[1]
    tokens_per_step = args.batch_size * world_size * seq_len

    logs = []
    step = 0
    data_iter = iter(loader)

    torch.cuda.reset_peak_memory_stats(device)
    model.train()

    while step < args.steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            sampler.set_epoch(step)
            data_iter = iter(loader)
            batch = next(data_iter)

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        torch.cuda.synchronize()
        t0 = time.perf_counter()

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        torch.cuda.synchronize()
        t1 = time.perf_counter()

        step += 1
        step_time = t1 - t0

        if step > args.warmup_steps and rank == 0:
            peak_mem_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
            logs.append({
                "step": step,
                "timestamp": time.time(),
                "step_time_sec": step_time,
                "tokens_per_sec": tokens_per_step / step_time,
                "peak_memory_mb": peak_mem_mb,
                "loss": loss.item(),
            })
            if step % 10 == 0:
                print(f"[step {step}] loss={loss.item():.4f} "
                      f"tok/s={tokens_per_step / step_time:.1f} "
                      f"peak_mem={peak_mem_mb:.0f}MB")

    if rank == 0:
        with open(args.out, "w") as f:
            json.dump({
                "framework": "pytorch_fsdp",
                "sharding_strategy": args.sharding_strategy,
                "world_size": world_size,
                "seq_len": seq_len,
                "per_gpu_batch_size": args.batch_size,
                "warmup_steps_excluded": args.warmup_steps,
                "logs": logs,
            }, f, indent=2)
        print(f"Saved results to {args.out}")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing pytorch_fsdp.py


# Qwen JAX implementation

In [4]:
%%writefile qwen2_flax.py
"""
Flax (JAX) implementation of the Qwen2 architecture.

transformers has no FlaxQwen2ForCausalLM, so this reimplements the pieces
needed to load Qwen/Qwen2.5-1.5B-Instruct's official PyTorch weights into
a JAX/Flax model with matching math.

Architecture notes:
  - RMSNorm, RoPE, GQA (with Qwen2's q/k/v bias), SwiGLU MLP.
  - Decoder layers are wrapped in nn.remat when cfg.use_remat is True:
    activations inside each layer are recomputed during backward instead
    of being stored. Costs ~30% extra compute, cuts activation memory by
    roughly the layer count. nn.remat preserves the variable tree, so the
    weight conversion and parity check are unaffected.
"""

from dataclasses import dataclass
from typing import Optional

import jax
import jax.numpy as jnp
import flax.linen as nn


@dataclass
class Qwen2Config:
    vocab_size: int = 151936
    hidden_size: int = 1536
    intermediate_size: int = 8960
    num_hidden_layers: int = 28
    num_attention_heads: int = 12
    num_key_value_heads: int = 2
    max_position_embeddings: int = 32768
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1000000.0
    tie_word_embeddings: bool = True
    use_remat: bool = True

    @classmethod
    def from_hf_config(cls, hf_config, use_remat: bool = True):
        """
        transformers v5 replaced the standalone `rope_theta` attribute with a
        `rope_parameters` dict. Older versions expose `rope_theta` directly.
        Handle both.
        """
        if hasattr(hf_config, "rope_theta"):
            rope_theta = hf_config.rope_theta
        elif hasattr(hf_config, "rope_parameters") and hf_config.rope_parameters:
            rp = hf_config.rope_parameters
            rope_theta = rp["rope_theta"] if isinstance(rp, dict) else rp.rope_theta
        else:
            rope_theta = 1000000.0

        return cls(
            vocab_size=hf_config.vocab_size,
            hidden_size=hf_config.hidden_size,
            intermediate_size=hf_config.intermediate_size,
            num_hidden_layers=hf_config.num_hidden_layers,
            num_attention_heads=hf_config.num_attention_heads,
            num_key_value_heads=hf_config.num_key_value_heads,
            max_position_embeddings=hf_config.max_position_embeddings,
            rms_norm_eps=hf_config.rms_norm_eps,
            rope_theta=rope_theta,
            tie_word_embeddings=hf_config.tie_word_embeddings,
            use_remat=use_remat,
        )


class RMSNorm(nn.Module):
    dim: int
    eps: float = 1e-6

    @nn.compact
    def __call__(self, x):
        weight = self.param("weight", nn.initializers.ones, (self.dim,))
        x_f32 = x.astype(jnp.float32)
        variance = jnp.mean(x_f32 * x_f32, axis=-1, keepdims=True)
        x_normed = x_f32 * jax.lax.rsqrt(variance + self.eps)
        return (weight * x_normed.astype(x.dtype))


def precompute_rope(head_dim: int, max_positions: int, theta: float):
    """cos/sin tables for RoPE, shape (max_positions, head_dim)."""
    inv_freq = 1.0 / (theta ** (jnp.arange(0, head_dim, 2, dtype=jnp.float32) / head_dim))
    positions = jnp.arange(max_positions, dtype=jnp.float32)
    freqs = jnp.outer(positions, inv_freq)
    emb = jnp.concatenate([freqs, freqs], axis=-1)
    return jnp.cos(emb), jnp.sin(emb)


def rotate_half(x):
    x1, x2 = jnp.split(x, 2, axis=-1)
    return jnp.concatenate([-x2, x1], axis=-1)


def apply_rope(q, k, cos, sin):
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


class Qwen2Attention(nn.Module):
    config: Qwen2Config

    @nn.compact
    def __call__(self, x, cos, sin, attn_mask):
        cfg = self.config
        B, T, C = x.shape
        n_heads = cfg.num_attention_heads
        n_kv_heads = cfg.num_key_value_heads
        head_dim = cfg.hidden_size // n_heads
        n_rep = n_heads // n_kv_heads

        # Qwen2 uses bias on q/k/v projections (unlike Llama).
        q = nn.Dense(n_heads * head_dim, use_bias=True, name="q_proj")(x)
        k = nn.Dense(n_kv_heads * head_dim, use_bias=True, name="k_proj")(x)
        v = nn.Dense(n_kv_heads * head_dim, use_bias=True, name="v_proj")(x)

        q = q.reshape(B, T, n_heads, head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, n_kv_heads, head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, n_kv_heads, head_dim).transpose(0, 2, 1, 3)

        q, k = apply_rope(q, k, cos[:T], sin[:T])

        k = jnp.repeat(k, n_rep, axis=1)
        v = jnp.repeat(v, n_rep, axis=1)

        scale = 1.0 / jnp.sqrt(head_dim).astype(x.dtype)
        attn_weights = jnp.einsum("bhtd,bhsd->bhts", q, k) * scale
        attn_weights = attn_weights + attn_mask
        attn_weights = jax.nn.softmax(attn_weights.astype(jnp.float32), axis=-1).astype(x.dtype)

        out = jnp.einsum("bhts,bhsd->bhtd", attn_weights, v)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, n_heads * head_dim)

        out = nn.Dense(cfg.hidden_size, use_bias=False, name="o_proj")(out)
        return out


class Qwen2MLP(nn.Module):
    config: Qwen2Config

    @nn.compact
    def __call__(self, x):
        cfg = self.config
        gate = nn.Dense(cfg.intermediate_size, use_bias=False, name="gate_proj")(x)
        up = nn.Dense(cfg.intermediate_size, use_bias=False, name="up_proj")(x)
        hidden = jax.nn.silu(gate) * up
        return nn.Dense(cfg.hidden_size, use_bias=False, name="down_proj")(hidden)


class Qwen2DecoderLayer(nn.Module):
    config: Qwen2Config

    @nn.compact
    def __call__(self, x, cos, sin, attn_mask):
        cfg = self.config
        residual = x
        x = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps, name="input_layernorm")(x)
        x = Qwen2Attention(cfg, name="self_attn")(x, cos, sin, attn_mask)
        x = residual + x

        residual = x
        x = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps, name="post_attention_layernorm")(x)
        x = Qwen2MLP(cfg, name="mlp")(x)
        x = residual + x
        return x


class Qwen2Model(nn.Module):
    config: Qwen2Config

    @nn.compact
    def __call__(self, input_ids, attention_mask):
        cfg = self.config
        B, T = input_ids.shape
        head_dim = cfg.hidden_size // cfg.num_attention_heads

        embed = nn.Embed(cfg.vocab_size, cfg.hidden_size, name="embed_tokens")
        x = embed(input_ids)

        # Only T positions are ever used. Building the full 32768-row table
        # inside the traced graph wastes ~33MB of fp32 constants per step;
        # cos[:T] of the long table is elementwise identical to this.
        cos, sin = precompute_rope(head_dim, T, cfg.rope_theta)

        causal = jnp.tril(jnp.ones((T, T), dtype=bool))
        pad = attention_mask[:, None, None, :].astype(bool)
        combined = causal[None, None, :, :] & pad
        attn_mask = jnp.where(combined, 0.0, jnp.finfo(x.dtype).min).astype(x.dtype)

        # nn.remat wraps the layer class; the variable tree (and therefore
        # the param names the conversion writes into) is unchanged.
        layer_cls = nn.remat(Qwen2DecoderLayer) if cfg.use_remat else Qwen2DecoderLayer

        for i in range(cfg.num_hidden_layers):
            x = layer_cls(cfg, name=f"layers_{i}")(x, cos, sin, attn_mask)

        x = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps, name="norm")(x)
        return x


class Qwen2ForCausalLM(nn.Module):
    config: Qwen2Config

    @nn.compact
    def __call__(self, input_ids, attention_mask):
        cfg = self.config
        hidden = Qwen2Model(cfg, name="model")(input_ids, attention_mask)
        logits = nn.Dense(cfg.vocab_size, use_bias=False, name="lm_head")(hidden)
        return logits

Writing qwen2_flax.py


In [5]:
%%writefile parity_check.py
"""
Loads Qwen/Qwen2.5-1.5B-Instruct's official PyTorch weights, converts them
into the Flax params pytree defined in qwen2_flax.py, and verifies the two
implementations produce matching logits on the same input.

Run this BEFORE any training/benchmarking. If parity fails, the JAX
benchmark numbers are meaningless -- fix the port first.

Usage:
    python parity_check.py
"""

import numpy as np
import jax.numpy as jnp
import torch
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer

from qwen2_flax import Qwen2Config, Qwen2ForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


def convert_pytorch_to_flax_params(pt_state_dict, cfg: Qwen2Config):
    """
    Maps HF PyTorch Qwen2 state_dict keys -> the nested param pytree Flax
    expects, matching the module names used in qwen2_flax.py.

    Key mapping notes:
      - PyTorch nn.Linear weight shape is (out_features, in_features).
        Flax nn.Dense kernel shape is (in_features, out_features).
        -> every Linear weight needs a transpose.
      - HF names layers as `model.layers.{i}.*`; qwen2_flax.py names them
        `layers_{i}` (underscore, not dot-index) -- mapped explicitly below.
    """
    # .float() upcast: PyTorch bf16 tensors have no numpy equivalent dtype,
    # so .numpy() on a bf16 tensor raises TypeError. Upcasting to fp32 here
    # is just to get through the numpy conversion -- train_jax.py casts
    # back down to bf16 itself after this returns.
    sd = {k: v.detach().float().cpu().numpy() for k, v in pt_state_dict.items()}
    params = {"model": {}, "lm_head": {}}

    params["model"]["embed_tokens"] = {"embedding": sd["model.embed_tokens.weight"]}
    params["model"]["norm"] = {"weight": sd["model.norm.weight"]}

    for i in range(cfg.num_hidden_layers):
        prefix = f"model.layers.{i}."
        layer_params = {
            "input_layernorm": {"weight": sd[prefix + "input_layernorm.weight"]},
            "post_attention_layernorm": {"weight": sd[prefix + "post_attention_layernorm.weight"]},
            "self_attn": {
                "q_proj": {
                    "kernel": sd[prefix + "self_attn.q_proj.weight"].T,
                    "bias": sd[prefix + "self_attn.q_proj.bias"],
                },
                "k_proj": {
                    "kernel": sd[prefix + "self_attn.k_proj.weight"].T,
                    "bias": sd[prefix + "self_attn.k_proj.bias"],
                },
                "v_proj": {
                    "kernel": sd[prefix + "self_attn.v_proj.weight"].T,
                    "bias": sd[prefix + "self_attn.v_proj.bias"],
                },
                "o_proj": {"kernel": sd[prefix + "self_attn.o_proj.weight"].T},
            },
            "mlp": {
                "gate_proj": {"kernel": sd[prefix + "mlp.gate_proj.weight"].T},
                "up_proj": {"kernel": sd[prefix + "mlp.up_proj.weight"].T},
                "down_proj": {"kernel": sd[prefix + "mlp.down_proj.weight"].T},
            },
        }
        params["model"][f"layers_{i}"] = layer_params

    if cfg.tie_word_embeddings:
        lm_head_weight = sd["model.embed_tokens.weight"]
    else:
        lm_head_weight = sd["lm_head.weight"]
    params["lm_head"] = {"kernel": lm_head_weight.T}

    def to_jnp(tree):
        if isinstance(tree, dict):
            return {k: to_jnp(v) for k, v in tree.items()}
        return jnp.array(tree)

    return {"params": to_jnp(params)}


def main():
    print(f"Loading PyTorch model + config: {MODEL_NAME}")
    hf_config = AutoConfig.from_pretrained(MODEL_NAME)
    pt_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
    pt_model.eval()

    cfg = Qwen2Config.from_hf_config(hf_config)
    print(f"Config: hidden={cfg.hidden_size}, layers={cfg.num_hidden_layers}, "
          f"heads={cfg.num_attention_heads}, kv_heads={cfg.num_key_value_heads}")

    print("Converting weights to Flax pytree ...")
    flax_params = convert_pytorch_to_flax_params(pt_model.state_dict(), cfg)

    # sanity check: total param count should match PyTorch's, given tying
    n_params = sum(x.size for x in jax.tree_util.tree_leaves(flax_params) if hasattr(x, "size"))
    pt_n_params = sum(p.numel() for p in pt_model.parameters())
    print(f"Flax param count: {n_params/1e9:.4f}B, PyTorch param count: {pt_n_params/1e9:.4f}B")

    flax_model = Qwen2ForCausalLM(cfg)

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    test_text = "The capital of France is"
    enc = tok(test_text, return_tensors="pt")
    input_ids_pt = enc["input_ids"]
    attention_mask_pt = enc["attention_mask"]

    print(f"Test input: {test_text!r} -> {input_ids_pt.shape[1]} tokens")

    with torch.no_grad():
        pt_out = pt_model(input_ids=input_ids_pt, attention_mask=attention_mask_pt)
        pt_logits = pt_out.logits.numpy().astype(np.float32)

    input_ids_jax = jnp.array(input_ids_pt.numpy())
    attention_mask_jax = jnp.array(attention_mask_pt.numpy())
    flax_logits = flax_model.apply(flax_params, input_ids_jax, attention_mask_jax)
    flax_logits = np.array(flax_logits).astype(np.float32)

    print(f"PyTorch logits shape: {pt_logits.shape}")
    print(f"Flax logits shape:    {flax_logits.shape}")

    abs_diff = np.abs(pt_logits - flax_logits)
    max_diff = abs_diff.max()
    mean_diff = abs_diff.mean()

    print(f"\nMax abs logit diff:  {max_diff:.6f}")
    print(f"Mean abs logit diff: {mean_diff:.6f}")

    pt_next_token = pt_logits[0, -1].argmax()
    flax_next_token = flax_logits[0, -1].argmax()
    pt_word = tok.decode([pt_next_token])
    flax_word = tok.decode([flax_next_token])
    print(f"\nPyTorch predicted next token: {pt_word!r}")
    print(f"Flax predicted next token:    {flax_word!r}")

    TOLERANCE = 0.05
    if max_diff < TOLERANCE and pt_next_token == flax_next_token:
        print(f"\n✅ PARITY CHECK PASSED (max_diff={max_diff:.6f} < {TOLERANCE}, "
              f"predicted tokens match)")
    else:
        print(f"\n❌ PARITY CHECK FAILED -- do NOT trust benchmark results from this port yet.")
        print("Likely causes: wrong transpose somewhere, RoPE base/formula mismatch, "
              "GQA head-repeat axis wrong, or tie_word_embeddings mismatch. "
              "Debug layer-by-layer (compare hidden states after layer 0) before re-running.")
        raise SystemExit(1)


if __name__ == "__main__":
    main()

Writing parity_check.py


# JAX

In [6]:
%%writefile train_jax.py
"""
JAX full-fine-tuning benchmark for the Flax Qwen2 port (qwen2_flax.py) on
Alpaca, using FSDP-style sharding across a device mesh.

TPU-adapted: n_devices/mesh code is backend-agnostic already; this version
just makes the memory-stats readout robust across TPU/GPU key differences
and drops the CUDA-only preallocate env var reference from the docstring.
"""

import os
import json
import time
import argparse
from functools import partial

import numpy as np
import jax
import jax.numpy as jnp
import optax
from jax.sharding import Mesh, PartitionSpec as P, NamedSharding
from transformers import AutoModelForCausalLM, AutoConfig
import torch

from qwen2_flax import Qwen2Config, Qwen2ForCausalLM
from parity_check import convert_pytorch_to_flax_params

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


def get_peak_mem_mb(devices):
    """
    TPU and GPU memory_stats() dicts have historically used different keys
    across JAX versions ('peak_bytes_in_use' on GPU; TPU sometimes reports
    only 'bytes_in_use' with no peak tracking). Try the known keys in order
    and fall back to current usage if no peak key exists, rather than crash.
    """
    vals = []
    for d in devices:
        stats = d.memory_stats()
        if stats is None:
            continue
        for key in ("peak_bytes_in_use", "bytes_in_use", "bytes_reserved"):
            if key in stats:
                vals.append(stats[key])
                break
    return max(vals) / (1024 ** 2) if vals else float("nan")


def load_sharded_params(cfg: Qwen2Config, mesh: Mesh):
    print("Loading PyTorch weights for conversion ...")
    pt_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
    params = convert_pytorch_to_flax_params(pt_model.state_dict(), cfg)
    del pt_model

    params = jax.tree_util.tree_map(lambda x: x.astype(jnp.bfloat16), params)

    sharding = NamedSharding(mesh, P("data"))

    def shard_leaf(x):
        if x.ndim == 0:
            return jax.device_put(x, NamedSharding(mesh, P()))
        return jax.device_put(x, sharding)

    return jax.tree_util.tree_map(shard_leaf, params)


def make_causal_lm_loss(model: Qwen2ForCausalLM):
    def loss_fn(params, batch):
        logits = model.apply(params, batch["input_ids"], batch["attention_mask"])
        logits = logits[:, :-1, :]
        labels = batch["labels"][:, 1:]
        mask = labels != -100
        labels_safe = jnp.where(mask, labels, 0)
        nll = optax.softmax_cross_entropy_with_integer_labels(
            logits.astype(jnp.float32), labels_safe
        )
        nll = jnp.where(mask, nll, 0.0)
        return nll.sum() / jnp.maximum(mask.sum(), 1)
    return loss_fn


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, default="tokenized_alpaca.npz")
    parser.add_argument("--batch-size", type=int, default=1, help="per-device micro batch size")
    parser.add_argument("--steps", type=int, default=100)
    parser.add_argument("--lr", type=float, default=1e-5)
    parser.add_argument("--no-remat", action="store_true",
                        help="disable gradient checkpointing (TPU has far more HBM, may not need it)")
    parser.add_argument("--out", type=str, default="results_jax.json")
    args = parser.parse_args()

    devices = jax.devices()
    n_devices = len(devices)
    print(f"JAX sees {n_devices} device(s): {devices}")
    print(f"Device platform: {devices[0].platform}")  # 'tpu' or 'gpu' -- confirms which backend actually loaded

    mesh = Mesh(np.array(devices), axis_names=("data",))

    hf_config = AutoConfig.from_pretrained(MODEL_NAME)
    cfg = Qwen2Config.from_hf_config(hf_config, use_remat=not args.no_remat)
    print(f"remat (gradient checkpointing): {cfg.use_remat}")
    model = Qwen2ForCausalLM(cfg)

    with mesh:
        params = load_sharded_params(cfg, mesh)

        loss_fn = make_causal_lm_loss(model)
        grad_fn = jax.value_and_grad(loss_fn)

        optimizer = optax.adamw(args.lr)
        opt_state = optimizer.init(params)

        leaf = jax.tree_util.tree_leaves(params)[0]
        n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
        print(f"params: {n_params/1e9:.3f}B, dtype={leaf.dtype}")
        print(f"leaf sharding: {leaf.sharding}")
        print(f"device 0 memory_stats keys: {list(devices[0].memory_stats().keys())}")
        print("peak/bytes-in-use after init: %.2f GB" % (get_peak_mem_mb([devices[0]]) / 1024))

        data_sharding = NamedSharding(mesh, P("data"))

        @partial(jax.jit, donate_argnums=(0, 1))
        def train_step(params, opt_state, batch):
            loss, grads = grad_fn(params, batch)
            updates, opt_state = optimizer.update(grads, opt_state, params)
            params = optax.apply_updates(params, updates)
            return params, opt_state, loss

        data = np.load(args.data)
        input_ids_all = data["input_ids"]
        attention_mask_all = data["attention_mask"]
        labels_all = data["labels"]
        n_examples = input_ids_all.shape[0]
        seq_len = input_ids_all.shape[1]

        global_batch_size = args.batch_size * n_devices
        tokens_per_step = global_batch_size * seq_len
        print(f"global_batch_size={global_batch_size} (per_device={args.batch_size} x n_devices={n_devices})")

        rng = np.random.default_rng(seed=42)
        logs = []
        compile_time_sec = None

        for step in range(1, args.steps + 1):
            idx = rng.choice(n_examples, size=global_batch_size, replace=False)
            batch = {
                "input_ids": jax.device_put(jnp.array(input_ids_all[idx]), data_sharding),
                "attention_mask": jax.device_put(jnp.array(attention_mask_all[idx]), data_sharding),
                "labels": jax.device_put(jnp.array(labels_all[idx]), data_sharding),
            }

            t0 = time.perf_counter()
            params, opt_state, loss = train_step(params, opt_state, batch)
            loss.block_until_ready()
            t1 = time.perf_counter()

            step_time = t1 - t0

            if step == 1:
                compile_time_sec = step_time
                print(f"[step 1] compile+run time: {compile_time_sec:.2f}s (excluded from steady-state stats)")
                continue

            peak_mem_mb = get_peak_mem_mb(devices)

            logs.append({
                "step": step,
                "timestamp": time.time(),
                "step_time_sec": step_time,
                "tokens_per_sec": tokens_per_step / step_time,
                "peak_memory_mb": peak_mem_mb,
                "loss": float(loss),
            })

            if step % 10 == 0:
                print(f"[step {step}] loss={float(loss):.4f} "
                      f"tok/s={tokens_per_step / step_time:.1f} "
                      f"peak_mem={peak_mem_mb:.0f}MB")

        with open(args.out, "w") as f:
            json.dump({
                "framework": "jax_fsdp_style",
                "platform": devices[0].platform,
                "n_devices": n_devices,
                "seq_len": seq_len,
                "per_device_batch_size": args.batch_size,
                "global_batch_size": global_batch_size,
                "remat": cfg.use_remat,
                "compile_time_sec": compile_time_sec,
                "logs": logs,
            }, f, indent=2)
        print(f"Saved results to {args.out}")


if __name__ == "__main__":
    main()

Writing train_jax.py


In [7]:
!python data_prep.py --n-examples 3000 --max-length 512

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 3.33MB/s]
tokenizer_config.json: 7.30kB [00:00, 20.5MB/s]
vocab.json: 2.78MB [00:00, 48.8MB/s]
merges.txt: 1.67MB [00:00, 112MB/s]
tokenizer.json: 7.03MB [00:00, 133MB/s]
Loading tatsu-lab/alpaca ...
README.md: 7.47kB [00:00, 26.2MB/s]
data/train-00000-of-00001-a09b74b3ef9c3b(…): 100%|█| 24.2M/24.2M [00:00<00:00, 3
Generating train split: 100%|██| 52002/52002 [00:00<00:00, 263047.24 examples/s]
Tokenizing 3000 examples, max_length=512 ...
Saved 3000 examples of length 512 to tokenized_alpaca.npz
Shapes -> input_ids: (3000, 512), attention_mask: (3000, 512), labels: (3000, 512)


In [8]:
import jax
print(jax.local_device_count())
print(jax.devices()[0].device_kind)

2
Tesla T4


In [10]:
!NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node=2 --standalone pytorch_ddp.py --data tokenized_alpaca.npz --steps 100

W0824 03:14:19.437000 307 torch/distributed/run.py:852] 
W0824 03:14:19.437000 307 torch/distributed/run.py:852] *****************************************
W0824 03:14:19.437000 307 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0824 03:14:19.437000 307 torch/distributed/run.py:852] *****************************************
[W824 03:14:19.767282806 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W824 03:14:27.416424527 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W824 03:14:27.441939562 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
World size: 2, loading model Qwen/Qwen2.5-1.5B-Instruct in bf16 ...
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use 

In [11]:
!NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node=2 --standalone pytorch_fsdp.py --data tokenized_alpaca.npz --steps 100

W0824 03:15:02.371000 358 torch/distributed/run.py:852] 
W0824 03:15:02.371000 358 torch/distributed/run.py:852] *****************************************
W0824 03:15:02.371000 358 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0824 03:15:02.371000 358 torch/distributed/run.py:852] *****************************************
[W824 03:15:02.771546693 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W824 03:15:10.667828937 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W824 03:15:10.737269255 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
World size: 2, loading model Qwen/Qwen2.5-1.5B-Instruct in bf16 ...
Sharding strategy: FULL_SHARD
`torch_dtype` is deprecated! Use `dtype` instead!
`to

In [9]:
!python train_jax.py --data tokenized_alpaca.npz --steps 100 --no-remat

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


JAX sees 2 device(s): [CudaDevice(id=0), CudaDevice(id=1)]
Device platform: gpu
remat (gradient checkpointing): False
Loading PyTorch weights for conversion ...
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:21<00:00, 145MB/s]
Loading weights: 100%|█| 338/338 [00:00<00:00, 1632.93it/s, Materializing param=
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 1.54MB/s]
params: 1.777B, dtype=bfloat16
leaf sharding: NamedSharding(mesh=Mesh('data': 2, axis_types=(Auto,)), spec=PartitionSpec('data',), memory_kind=device)
device 0 memory_stats keys: ['num_allocs', 'bytes_in_use', 'peak_bytes_in_use', 'largest_alloc_size', 'bytes_limit', 'bytes_reserved', 'peak_bytes_reserved', 'largest_free_block_bytes', 'pool_bytes', 'peak_pool_bytes']
peak/bytes-in-use after init: 10.03 GB
global_batch_size=2 (per_device=1 x n_devices=2)
[step 1] compile+run time: 52.83s (excluded from steady-state stats)
[step 10] loss=1.64